# Analitycal model of Google's MICROSERVICES DEMO

This notebook requires some json file to build the jackson network and solve the traffic equations. In particular we will need:

1. overall_avg_metrics-low_load.json collected in low load to estimate service rates for leaf services.
2. routing_count.json for building the routing matrix and solving the traffic equations.
3. overall_avg_metrics-normal_load.json to measure empirical metrics under experimental load for comparisons with model predictions.
4. upstream_response_time-low_load.json to estimate service rates for upstream nodes using the traces method.
5. upstream_response_time-normal_load.json to measure the response time of upstream services under experimental arrival rates using the traces method.
6. emp_calls_count.json to compute empirical response time with the metrics method under normal load.
7. calls_count.json to estimate service time with the metrics method under low load.
8. emp_routing_count.json to help in computing empirical response time with the metrics method under normal load.

The notebook will be able to conclude whether with the currenct external arrival rate the system will be able to reach a steady state assuming M/M/1 queueing behavior for all nodes. In the case a steady state is reachable the notebook will compute staedy state performance metrics and compare them with actual measurements.


## Estimating service rates
Since we have not collected CPU consumption metrics (which would have allowed us to calculate the service rate $\mu$ by dividing throughput X with CPU utilization U), we will need to estimate service time by collecting the average response time in low load scenarios. However, just doing this is not enough, since the response time of upstream services also account for the response time of relied upon downstream services, given the way we collected the metric. Because of this we will have to rely on trace data to estimate average service time by looking at the difference between the start time of a span for an upstream service and the start time for the next downstream service span.

$R_i \approx S_i$, then $\mu_i \approx \frac{1}{R_i}$

The other way to estimate service time is to subtract the response timefrom downstream services through the formula:

$R_i=r_i-\sum_{j=0}^{N}r_j*n_{ij}\approx S_i$ where $r_i$ is the registered metric at node i and $n_{ij}$ is the average number of calls from i to j for i call.

In [151]:
# load json data
import json
import numpy as np

with open("../jupyter_data/overall_avg_metrics-low_load.json", "r") as f:
    metrics_dict = json.load(f)

with open("../jupyter_data/routing_count.json","r") as f:
    routing_dict=json.load(f)

with open("../jupyter_data/upstream_response_time-low_load.json", "r") as f:
    upstream_resp_dict=json.load(f)

with open("../jupyter_data/calls_count.json") as f:
    call_count_dict=json.load(f)

other_endpoints = [
    "GetAds",
    "ListRecommendations",
    "GetCart",
    "AddItem",
    "EmptyCart",
    "PlaceOrder",
    "Charge",
    "GetQuote",
    "ShipOrder",
    "SendOrderConfirmation",
    "GetSupportedCurrencies",
    "Convert",
    "ListProducts",
    "GetProduct",
]
services = [
    "frontend",
    "adservice",
    "recommendationservice",
    "cartservice",
    "checkoutservice",
    "paymentservice",
    "shippingservice",
    "emailservice",
    "currencyservice",
    "productcatalogservice",
]

metrics = [
    "requests_total",
    "requests_duration",
    "active_requests",
]

In [152]:
# Compute approximated service rate and save it in a json file using tracing

service_rate = {service: 0 for service in services}

for service in services:

    R_i=upstream_resp_dict[service]

    mu_i = 1 / R_i
    service_rate[service] = mu_i

print("Service rates for each service computed with collected traces:")

for service, rate in service_rate.items():
    print(f"{service} - {rate}")

Service rates for each service computed with collected traces:
frontend - 1081.9230000716905
adservice - 466.64630432650256
recommendationservice - 448.79627096852084
cartservice - 352.7325352813
checkoutservice - 1546.0611637898162
paymentservice - 359.1265507073078
shippingservice - 759.2274671521805
emailservice - 228.13354183322042
currencyservice - 173.3263063797736
productcatalogservice - 630.0628756438136


In [153]:
# Compute approximated service rate and save it in a json file using metrics
service_rate = {service: 0 for service in services}
for service_i, metric_dict in metrics_dict.items():
    r_i=metric_dict["requests_duration"]
    sum=0

    for servide_j, call_count_value in call_count_dict[service_i].items():
        sum+=call_count_value*metrics_dict[servide_j]["requests_duration"]

    if service_i == "frontend":
        sum=sum/routing_dict["frontend"]["in"]

    if service_i == "recommendationservice":
        sum=sum/call_count_dict["frontend"]["recommendationservice"]

    if service_i == "checkoutservice":
        sum=sum/call_count_dict["frontend"]["checkoutservice"]
    
    R_i=r_i-sum
    mu_i=1/R_i
    service_rate[service_i] = mu_i

print("Service rates for each service computed with collected metrics:")

for service, rate in service_rate.items():
    print(f"{service} - {rate}")

Service rates for each service computed with collected metrics:
frontend - 31.330696985264034
adservice - 3954.8022598870025
recommendationservice - 284.7289874633431
cartservice - 1096.1763411234535
checkoutservice - 52.953086851664615
paymentservice - 3285.714285714285
shippingservice - 11954.708939153048
emailservice - 3384.3227541689084
currencyservice - 4613.916500994038
productcatalogservice - 15342.389173555272


## Solving Traffic Equations

Given the routing count of departing requests on each node of the Queueing network (used to compute routing probabilities) and the external arrival rate $\gamma$ for the frontend, we can solve the Traffic Equations and obtain the aggregate arrival rate $\lambda_i$ for each service i.

$\lambda_i = \gamma_i + \sum_{j=0}^{N-1}q_{ji}\lambda_j$ 

In the matrix form we have:

If $\underline{\lambda}=[\lambda_1, \lambda_2, ..., \lambda_N]^T$ and $\underline{\gamma}=[\gamma_1, \gamma_2, ..., \gamma_N]^T$, then $\underline{\lambda}=\underline{\gamma}+Q^T\underline{\lambda}$ which implies $(I-Q^T)\underline{\lambda}=\underline{\gamma}$

In [154]:
# Run to build the external arrival rate vector
service_index_map = {
        "frontend": 0,
        "adservice": 1,
        "recommendationservice": 2,
        "cartservice": 4,
        "checkoutservice": 8,
        "paymentservice": 7,
        "shippingservice": 5,
        "emailservice": 9,
        "currencyservice": 6,
        "productcatalogservice": 3,
    }

function_map = {
    "GetAds": "adservice",
    "ListRecommendations": "recommendationservice",
    "GetCart": "cartservice",
    "AddItem": "cartservice",
    "EmptyCart": "cartservice",
    "PlaceOrder": "checkoutservice",
    "Charge": "paymentservice",
    "GetQuote": "shippingservice",
    "ShipOrder": "shippingservice",
    "SendOrderConfirmation": "emailservice",
    "GetSupportedCurrencies": "currencyservice",
    "Convert": "currencyservice",
    "ListProducts": "productcatalogservice",
    "GetProduct": "productcatalogservice",
}
# Set the value you see from the collected metrics on the frontend in the empirical experiment
EXTERNAL_ARRIVAL_RATE=24.0

gamma=[0.0 for _ in range(10)]
gamma[0]=EXTERNAL_ARRIVAL_RATE
gamma=np.array(gamma).reshape(-1,1)


In [155]:
# Run to show gamma
print("External arrival rates for each service:")

for key, value in service_index_map.items():
        print(f"{key} - {gamma[value][0]}")

External arrival rates for each service:
frontend - 24.0
adservice - 0.0
recommendationservice - 0.0
cartservice - 0.0
checkoutservice - 0.0
paymentservice - 0.0
shippingservice - 0.0
emailservice - 0.0
currencyservice - 0.0
productcatalogservice - 0.0


In [156]:
# Run to build the routing matrix
Q=np.array([[0.0 for _ in range(10)] for _ in range(10)])

for service in services:
    total=0
    for name, value in routing_dict[service].items():
        if name=="in":
            continue
        if name=="out":
            total+=value
            continue
        total+=value
        Q[service_index_map[service],service_index_map[function_map[name]]]+=value
    Q[service_index_map[service],:]=Q[service_index_map[service],:]/total

In [157]:
# Run to show Q
print("Full routing matrix:")

for key, value in service_index_map.items():
        print(f"{key} - {value}")
print()
print("\t", end="")
for i in range(10):
    print(f"{i}", end="\t")
print()
for i in range(10):
    print(i, end="\t")
    for j in range(10):
        print(f"{Q[i,j]:.3f}", end="\t")
    print()

Full routing matrix:
frontend - 0
adservice - 1
recommendationservice - 2
cartservice - 4
checkoutservice - 8
paymentservice - 7
shippingservice - 5
emailservice - 9
currencyservice - 6
productcatalogservice - 3

	0	1	2	3	4	5	6	7	8	9	
0	0.000	0.000	0.000	0.502	0.032	0.000	0.342	0.000	0.031	0.000	
1	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	
2	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	0.000	
3	0.000	0.088	0.000	0.671	0.054	0.040	0.147	0.000	0.000	0.000	
4	0.000	0.000	0.198	0.033	0.000	0.000	0.602	0.000	0.000	0.033	
5	0.000	0.000	0.000	0.000	0.125	0.000	0.875	0.000	0.000	0.000	
6	0.000	0.051	0.133	0.122	0.194	0.010	0.409	0.010	0.000	0.000	
7	0.000	0.000	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	
8	0.000	0.000	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	
9	0.000	0.000	1.000	0.000	0.000	0.000	0.000	0.000	0.000	0.000	


In [158]:
# Run to solve for lambda and print results

aggregate_arrival_rates = np.linalg.solve(np.eye(10) - Q.T, gamma)

print("Aggregate arrival rates for each service:")

for key, value in service_index_map.items():
        print(f"{key} - {aggregate_arrival_rates[value][0]}")

Aggregate arrival rates for each service:
frontend - 24.0
adservice - 13.559524316951531
recommendationservice - 15.008567409743819
cartservice - 22.512595371041293
checkoutservice - 0.7406333915860364
paymentservice - 0.7406333915860361
shippingservice - 5.943480670048161
emailservice - 0.7406333915860361
currencyservice - 73.48484719321422
productcatalogservice - 111.74050551979877


## Compute Performance Metrics

We assign different queueing models to the service nodes and analyze if the network can reach the steady state under current arrival rate. If it can then we compute performance metrics at steady state considering two different modeling scenarios:

- Mean queue length at M/M/1 node i: $L_i=\frac{\rho_i}{1-\rho_i}$, with $\rho_i=\frac{\lambda_i}{\mu_i}$
- Mean queue length at M/M/inf node i: $L_i=\rho_i$, with $\rho_i=\frac{\lambda_i}{\mu_i}$
- Mean queue length at M/M/c node i: $L_i=\frac{\rho_i}{1-\rho_i}*C(c_i,\frac{\lambda_i}{\mu_i})+c_i*\rho$, with $\rho_i=\frac{\lambda_i}{c_i*\mu_i}$ and $C(c,\frac{\lambda}{\mu})$ the Erlang's C formula.
- Mean system queue length: $L=L_1+L_2+...+L_N=\sum_{i=0}^{N-1}\frac{\rho_i}{1-\rho_i}$
- Mean system response time: $W=\frac{L}{\gamma}$, where $\gamma=\sum_{i=0}^{N-1}\gamma_i$
- Mean response time at node i: $W_i=\frac{L_i}{\lambda_i}$

In [159]:
# Run to compute utilization for every node and then print it
import math
mu=np.zeros((10,1),float)

for service in services:
        mu[service_index_map[service]]=service_rate[service]

rho=np.divide(aggregate_arrival_rates, mu)
print("Rho for each service::")
for key, value in service_index_map.items():
        print(f"{key} - {rho[value][0]}")

Rho for each service::
frontend - 0.7660218989474786
adservice - 0.0034286225772863187
recommendationservice - 0.05271176476780773
cartservice - 0.020537384840808092
checkoutservice - 0.013986595222687269
paymentservice - 0.00022541016265661975
shippingservice - 0.000497166488979299
emailservice - 0.0002188424229555831
currencyservice - 0.015926783065402763
productcatalogservice - 0.0072831228732223125


In [160]:
# Run to define function to compute average queue length for M/M/1 servers
def M_M_1_queue_length(rho):
    return rho/(1.0-rho)

In [161]:
# Run to define function to compute average queue length for M/M/inf servers
def M_M_inf_queue_length(rho):
    return rho

In [162]:
# Run to define function to compute the Erlang's C formula
def Erlang_c(c, rho):
    rho_c=rho/c
    sum=0
    for k in range(c):
        sum+=rho**k/math.factorial(k)
    return 1/(1+(1-rho_c)*(math.factorial(c)/rho**c)*sum)

In [163]:
# Run to define function to compute average queue length for M/M/c servers
def M_M_c_queue_length(rho, c):
    rho_c=rho/c
    return rho_c*Erlang_c(c,rho)/(1-rho_c)+rho

In [164]:
# Run to check if the system can actually handle the load assuming the number of server to be equivalent to the cuncurrent behavior of the services.

steady_state=True

for service in services:
    if service in ["frontend", "checkoutservice", "adservice", "shippingservice", "cartservice", "productcatalogservice"]:
        continue
    elif service in ["currencyservice", "emailservice", "recommendationservice"]:
        if aggregate_arrival_rates[service_index_map[service]] > 10 * mu[service_index_map[service]]:
            print(f"Steady state does not exist, in {service} lambda is bigger than mu*c: {aggregate_arrival_rates[i,0]}>{mu[i,0]}")
            steady_state=False
            break
    else:
        if aggregate_arrival_rates[service_index_map[service]] > mu[service_index_map[service]]:
            print(f"Steady state does not exist, in {service} lambda is bigger than mu: {aggregate_arrival_rates[i,0]}>{mu[i,0]}")
            steady_state=False
            break

if steady_state:
    print("Steady state is reachable, we can now proceed to compute the steady state performance of the system.")

Steady state is reachable, we can now proceed to compute the steady state performance of the system.


In [165]:
# Run to heck if the system can actually handle the load assuming universal M/M/1 behavior
steady_state=True

for service in services:
    if aggregate_arrival_rates[service_index_map[service]] >  mu[service_index_map[service]]:
            print(f"Steady state does not exist, in {service} lambda is bigger than mu: {aggregate_arrival_rates[service_index_map[service],0]}>{mu[service_index_map[service],0]}")
            steady_state=False
            break

if steady_state:
    print("Steady state is reachable, we can now proceed to compute the steady state performance of the system.")

Steady state is reachable, we can now proceed to compute the steady state performance of the system.


In [166]:
# Run to compute mean queue length for every node and the mean system queue length given the second assumption (multiple types of queues), then print them.
L=np.zeros((10,1),float)
for service in services:
        if service in ["frontend", "checkoutservice", "adservice", "shippingservice", "cartservice", "productcatalogservice"]:
                L[service_index_map[service],0]=M_M_inf_queue_length(rho[service_index_map[service],0])
        elif service in ["currencyservice", "emailservice", "recommendationservice"]:
                L[service_index_map[service],0]=M_M_c_queue_length(rho[service_index_map[service],0],10)
        else:
                L[service_index_map[service],0]=M_M_1_queue_length(rho[service_index_map[service],0])

L_tot=np.sum(L)
print("Mean queue length for each service::")
for key, value in service_index_map.items():
        print(f"{key} - {L[value][0]}")
print(f"Mean number of requests in the system: ")
print(L_tot)

Mean queue length for each service::
frontend - 0.7660218989474786
adservice - 0.0034286225772863187
recommendationservice - 0.05271176476780773
cartservice - 0.020537384840808092
checkoutservice - 0.013986595222687269
paymentservice - 0.00022546098385366293
shippingservice - 0.000497166488979299
emailservice - 0.0002188424229555831
currencyservice - 0.015926783065402763
productcatalogservice - 0.0072831228732223125
Mean number of requests in the system: 
0.8808376421904816


In [167]:
# Run to compute mean queue length for every node and the mean system queue length given the first assumption (all M/M/1 queues), then print them.
L=np.zeros((10,1),float)
for service in services:
    L[service_index_map[service],0]=M_M_1_queue_length(rho[service_index_map[service],0])
L_tot=np.sum(L)
print("Mean queue length for each service::")
for key, value in service_index_map.items():
        print(f"{key} - {L[value][0]}")
print(f"Mean number of requests in the system: ")
print(L_tot)

Mean queue length for each service::
frontend - 3.273904247883133
adservice - 0.003440418473740699
recommendationservice - 0.055644905961370256
cartservice - 0.020968012992992238
checkoutservice - 0.014184995006073054
paymentservice - 0.00022546098385366293
shippingservice - 0.0004974137864450758
emailservice - 0.00021889032544476495
currencyservice - 0.01618455089654297
productcatalogservice - 0.00733655591139124
Mean number of requests in the system: 
3.392605452220987


In [168]:
# Run to compute mean system response time and print the results
gamma_tot=np.sum(gamma)
W_tot=L_tot/gamma_tot
print("Mean system response time: ")
print(W_tot)

Mean system response time: 
0.14135856050920778


In [169]:
# Run to compute response time at node i and show the results
W=np.divide(L,aggregate_arrival_rates)
print("Mean response time for each service:: ")
for key, value in service_index_map.items():
        print(f"{key} - {W[value][0]}")

Mean response time for each service:: 
frontend - 0.13641267699513054
adservice - 0.00025372707724264607
recommendationservice - 0.0037075427948735887
cartservice - 0.0009313903016248449
checkoutservice - 0.019152518867258283
paymentservice - 0.0003044164446472599
shippingservice - 8.369065435877747e-05
emailservice - 0.000295544770100117
currencyservice - 0.00022024337689631194
productcatalogservice - 6.565708538065734e-05


## Comparisons with empirically collected data

Like previously said, the metrics we have collected for upstream services also comprehend downstream contributions. For example in the frontend service the mean duration is just the response time for the entire system, while the mean active requests represents the mean user number in the system. We will derive the corresponding empirical metrics in the following way:

- Empirical Average System Response Time: just the mean response time at the frontend.
- Empirical Average Number of Requests In The System: just the mean active requests number in the frontend.
- Empirical Average System Arrival Rate: just the mean number of arrival of requests per second in the frontend.
- Empirical Response Time At Service i: for leaf services they correspond to their measured local response time, while for upstream services we compute them from traces.
- Empirical Average Service Arrival Rates: we already got that as the local arrival rate at each service.
- Empirical Queue Length at Service i: estimated using Little's Law on the singular queues for upstream services, while for leaf services it corresponds to their mean active requests metric.


In [170]:
# Run to see the empirical average system arrival rate at steady state
with open("../jupyter_data/service_average_metrics-normal_load.json", "r") as f:
    normal_metrics_dict=json.load(f)

with open("../jupyter_data/upstream_response_time-normal_load.json", "r") as f:
    normal_upstream_resp_dict=json.load(f)

with open("../jupyter_data/emp_routing_count.json", "r") as f:
    emp_routing_dict=json.load(f)

with open("../jupyter_data/emp_calls_count.json", "r") as f:
    emp_call_count_dict=json.load(f)

print(f"Average system arrival rate: {normal_metrics_dict["frontend"]["requests_rate"][-1][0]}")
print(f"System arrival rate used in the model: {np.sum(gamma)}")

Average system arrival rate: 23.999975331649704
System arrival rate used in the model: 24.0


In [171]:
# Run to see the empirical average number of requests in the system

print(f"Average system queue length {normal_metrics_dict["frontend"]["active_requests"][-1][0]}")
print(f"Predicted system queue length: {L_tot}")

Average system queue length 3.499743991142907
Predicted system queue length: 3.392605452220987


In [172]:
# Run to see the empirical average system response time

print(f"Average system response time {normal_metrics_dict["frontend"]["requests_duration"][-1][0]}")
print(f"Predicted system response time: {W_tot}")

Average system response time 0.15020808205488256
Predicted system response time: 0.14135856050920778


In [173]:
# Run to compute and show the empirical response time for each node i using trace data

response_time = {service: 0 for service in services}

for service in services:

    response_time[service] =normal_upstream_resp_dict[service]

print("Response time at each service node (measured using traces) compared with model prediction")

W_emp=np.zeros(10,float)

for service, rsp_time in response_time.items():
    print(f"{service}\t\t predicted response time: {W[service_index_map[service]][0]}, actual response time: {rsp_time}")
    W_emp[service_index_map[service]]=rsp_time

Response time at each service node (measured using traces) compared with model prediction
frontend		 predicted response time: 0.13641267699513054, actual response time: 0.0018791788410818973
adservice		 predicted response time: 0.00025372707724264607, actual response time: 0.004761991774251851
recommendationservice		 predicted response time: 0.0037075427948735887, actual response time: 0.024716425282440887
cartservice		 predicted response time: 0.0009313903016248449, actual response time: 0.01708629736807509
checkoutservice		 predicted response time: 0.019152518867258283, actual response time: 0.002557267208357115
paymentservice		 predicted response time: 0.0003044164446472599, actual response time: 0.003736689284041121
shippingservice		 predicted response time: 8.369065435877747e-05, actual response time: 0.007261902791685427
emailservice		 predicted response time: 0.000295544770100117, actual response time: 0.02928037498448346
currencyservice		 predicted response time: 0.000220243376

In [174]:
# Run to compute and show the empirical response time for each node i using metric data

response_time = {service: 0 for service in services}
for service_i, metric_dict in normal_metrics_dict.items():
    r_i=metric_dict["requests_duration"][-1][0]
    sum=0

    for servide_j, call_count_value in emp_call_count_dict[service_i].items():
        sum+=call_count_value*normal_metrics_dict[servide_j]["requests_duration"][-1][0]

    if service_i == "frontend":
        sum=sum/emp_routing_dict["frontend"]["in"]

    if service_i == "recommendationservice":
        sum=sum/emp_call_count_dict["frontend"]["recommendationservice"]

    if service_i == "checkoutservice":
        sum=sum/emp_call_count_dict["frontend"]["checkoutservice"]
    
    response_time[service_i] = r_i-sum

print("Response time at each service node (measured using metrics) compared with model prediction")

W_emp=np.zeros(10,float)

for service, rsp_time in response_time.items():
    print(f"{service}\t\t predicted response time: {W[service_index_map[service]][0]}, actual response time: {rsp_time}")
    W_emp[service_index_map[service]]=rsp_time

Response time at each service node (measured using metrics) compared with model prediction
frontend		 predicted response time: 0.13641267699513054, actual response time: 0.12925001130329195
adservice		 predicted response time: 0.00025372707724264607, actual response time: 0.0001837088388214379
recommendationservice		 predicted response time: 0.0037075427948735887, actual response time: 0.024990401506335785
cartservice		 predicted response time: 0.0009313903016248449, actual response time: 0.0013035479371256988
checkoutservice		 predicted response time: 0.019152518867258283, actual response time: 0.056724817658354215
paymentservice		 predicted response time: 0.0003044164446472599, actual response time: 0.00024242424242424264
shippingservice		 predicted response time: 8.369065435877747e-05, actual response time: 0.00034929432941177194
emailservice		 predicted response time: 0.000295544770100117, actual response time: 0.0006237983703613281
currencyservice		 predicted response time: 0.0002

In [175]:
# Run to show the empirical arrival rate for each node
lambda_emp=np.zeros(10,float)
print("Arrival rate at each service node compared with model prediction")
for service in services:
    print(f"{service}\t\t predicted arrival rate: {aggregate_arrival_rates[service_index_map[service]][0]}, actual arrival rate: {normal_metrics_dict[service]["requests_rate"][-1][0]}")
    lambda_emp[service_index_map[service]]=normal_metrics_dict[service]["requests_rate"][-1][0]

Arrival rate at each service node compared with model prediction
frontend		 predicted arrival rate: 24.0, actual arrival rate: 23.999975331649704
adservice		 predicted arrival rate: 13.559524316951531, actual arrival rate: 12.82221440975053
recommendationservice		 predicted arrival rate: 15.008567409743819, actual arrival rate: 14.527731714231518
cartservice		 predicted arrival rate: 22.512595371041293, actual arrival rate: 22.26668321601593
checkoutservice		 predicted arrival rate: 0.7406333915860364, actual arrival rate: 0.777787329944138
paymentservice		 predicted arrival rate: 0.7406333915860361, actual arrival rate: 0.7333007445513127
shippingservice		 predicted arrival rate: 5.943480670048161, actual arrival rate: 5.688862605042753
emailservice		 predicted arrival rate: 0.7406333915860361, actual arrival rate: 0.7774332284284973
currencyservice		 predicted arrival rate: 73.48484719321422, actual arrival rate: 69.90489742245114
productcatalogservice		 predicted arrival rate: 111.7

In [178]:
# Run to compute and show empirical queue length for each node

L_emp=np.zeros(10,float)

for service, metric_dict in normal_metrics_dict.items():
    
    L_i = normal_metrics_dict[service]["requests_rate"][-1][0]*response_time[service]

    L_emp[service_index_map[service]]=L_i

print("Queue length at each service node compared with model prediction")

for service in services:
    print(f"{service}\t\t predicted queue length: {L[service_index_map[service]][0]}, actual queue length: {L_emp[service_index_map[service]]}")

Queue length at each service node compared with model prediction
frontend		 predicted queue length: 3.273904247883133, actual queue length: 3.101997082894452
adservice		 predicted queue length: 0.003440418473740699, actual queue length: 0.0023555541203347788
recommendationservice		 predicted queue length: 0.055644905961370256, actual queue length: 0.36305384851497347
cartservice		 predicted queue length: 0.020968012992992238, actual queue length: 0.029025688972868988
checkoutservice		 predicted queue length: 0.014184995006073054, actual queue length: 0.044119844468059416
paymentservice		 predicted queue length: 0.00022546098385366293, actual queue length: 0.00017776987746698504
shippingservice		 predicted queue length: 0.0004974137864450758, actual queue length: 0.0019870874487441146
emailservice		 predicted queue length: 0.00021889032544476495, actual queue length: 0.00048496158095844276
currencyservice		 predicted queue length: 0.01618455089654297, actual queue length: 0.019753799684